# Resilience, Node Failure, and Distributed Risk Lab


## Purpose

This lab estimates node-level deployment feasibility and distributed risk.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

nodes = pd.DataFrame({
    "node_id": [f"edge_node_{i:02d}" for i in range(1, 9)],
    "local_examples": rng.integers(250, 1800, size=8),
    "edge_inference_ms": rng.uniform(18, 70, size=8),
    "local_action_ms": rng.uniform(8, 25, size=8),
    "uplink_ms": rng.uniform(35, 90, size=8),
    "cloud_inference_ms": rng.uniform(25, 45, size=8),
    "downlink_ms": rng.uniform(30, 80, size=8),
    "raw_bandwidth_mb_s": rng.uniform(20, 150, size=8),
    "edge_output_mb_s": rng.uniform(1, 8, size=8),
    "node_trust": rng.uniform(0.70, 0.98, size=8),
    "governance_review_complete": rng.choice([0, 1], size=8, p=[0.25, 0.75]),
})

In [ ]:
nodes["compute_capacity"] = rng.uniform(0.45, 0.95, size=len(nodes))
nodes["model_compute_demand"] = rng.uniform(0.30, 0.90, size=len(nodes))
nodes["memory_capacity"] = rng.uniform(0.40, 0.95, size=len(nodes))
nodes["model_memory_demand"] = rng.uniform(0.25, 0.90, size=len(nodes))
nodes["energy_budget"] = rng.uniform(0.45, 0.95, size=len(nodes))
nodes["model_energy_demand"] = rng.uniform(0.25, 0.88, size=len(nodes))

nodes["compute_feasible"] = nodes["model_compute_demand"] <= nodes["compute_capacity"]
nodes["memory_feasible"] = nodes["model_memory_demand"] <= nodes["memory_capacity"]
nodes["energy_feasible"] = nodes["model_energy_demand"] <= nodes["energy_budget"]

nodes["edge_latency_ms"] = nodes["edge_inference_ms"] + nodes["local_action_ms"]
nodes["latency_feasible"] = nodes["edge_latency_ms"] <= 100

nodes["deployment_feasible"] = (
    nodes["compute_feasible"]
    & nodes["memory_feasible"]
    & nodes["energy_feasible"]
    & nodes["latency_feasible"]
)

nodes["distributed_risk"] = (
    0.35 * (1 - nodes["node_trust"])
    + 0.30 * (1 - nodes["governance_review_complete"])
    + 0.35 * (~nodes["deployment_feasible"]).astype(int)
)

nodes[["node_id", "deployment_feasible", "node_trust", "distributed_risk"]].sort_values(
    "distributed_risk",
    ascending=False,
)

## Interpretation

Distributed risk increases when node trust, deployment feasibility, or governance review coverage is weak.